<a href="https://colab.research.google.com/github/liuxiaohu0511/lance-demo/blob/develop/lance_read_write.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pylance


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 20.9 MB/s eta 0:00:00


In [2]:
import shutil
import lance
import numpy as np
import pandas as pd
import pyarrow as pa

In [31]:
import lance
import pyarrow as pa

shutil.rmtree("./alice_and_bob.lance", ignore_errors=True)
table = pa.Table.from_pylist([{"name": "Alice", "age": 20},
                              {"name": "Bob", "age": 30}])
ds = lance.write_dataset(table, "./alice_and_bob.lance")

In [7]:
ds.versions()

[{'version': 1,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 10, 23, 886387),
  'metadata': {'total_data_file_rows': '2',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '593',
   'total_fragments': '1',
   'total_rows': '2'}}]

In [32]:
ds = lance.dataset("./alice_and_bob.lance")

In [33]:
ds.versions()

[{'version': 1,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 24, 58, 757445),
  'metadata': {'total_data_file_rows': '2',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '593',
   'total_fragments': '1',
   'total_rows': '2'}}]

当数据量特别大的时候，可以使用流式的方式写入

In [13]:

import pyarrow as pa  # 用于处理RecordBatch和schema
from pyarrow import dataset as pa_ds  # 仅用于类型提示（可选）
from typing import Iterator  # 用于指定生产者函数的返回类型
import lance  # 用于写入lance数据集
def producer() -> Iterator[pa.RecordBatch]:
    """An iterator of RecordBatches."""
    yield pa.RecordBatch.from_pylist([{"name": "Alice", "age": 20}])
    yield pa.RecordBatch.from_pylist([{"name": "Bob", "age": 30}])

schema = pa.schema([
    ("name", pa.string()),
    ("age", pa.int32()),
])

ds = lance.write_dataset(producer(),
                         "./alice_and_bob.lance",
                         schema=schema, mode="overwrite")
print(ds.count_rows())  # Output: 2

2


添加行

要将数据插入数据集，您可以使用 LanceDataset.insert 或 lance.write_dataset

In [14]:
import lance
import pyarrow as pa
shutil.rmtree("./insert_example.lance", ignore_errors=True)
table = pa.Table.from_pylist([{"name": "Alice", "age": 20},
                              {"name": "Bob", "age": 30}])
ds = lance.write_dataset(table, "./insert_example.lance")

new_table = pa.Table.from_pylist([{"name": "Carla", "age": 37}])
ds.insert(new_table)
print(ds.to_table().to_pandas())
#     name  age
# 0  Alice   20
# 1    Bob   30
# 2  Carla   37

new_table2 = pa.Table.from_pylist([{"name": "David", "age": 42}])
ds = lance.write_dataset(new_table2, ds, mode="append")
print(ds.to_table().to_pandas())
#     name  age
# 0  Alice   20
# 1    Bob   30
# 2  Carla   37
# 3  David   42

    name  age
0  Alice   20
1    Bob   30
2  Carla   37
    name  age
0  Alice   20
1    Bob   30
2  Carla   37
3  David   42


删除行

Lance 支持使用 SQL 过滤器从数据集中删除行，具体操作请参见过滤器下推 。例如，要从上述数据集中删除 Bob 的行，可以使用：

In [15]:
import lance

dataset = lance.dataset("./alice_and_bob.lance")
#这里删除name是Blob的行
dataset.delete("name = 'Bob'")
dataset2 = lance.dataset("./alice_and_bob.lance")
print(dataset2.to_table().to_pandas())
#     name  age
# 0  Alice   20

    name  age
0  Alice   20


In [16]:
dataset.versions()

[{'version': 1,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 10, 23, 886387),
  'metadata': {'total_data_file_rows': '2',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '593',
   'total_fragments': '1',
   'total_rows': '2'}},
 {'version': 2,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 14, 39, 929289),
  'metadata': {'total_data_file_rows': '2',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '593',
   'total_fragments': '1',
   'total_rows': '2'}},
 {'version': 3,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 17, 48, 970925),
  'metadata': {'total_data_file_rows': '2',
   'total_data_files': '1',
   'total_deletion_file_rows': '1',
   'total_deletion_files': '1',
   'total_files_size': '593',
   'total_fragments': '1',
   'total_rows': '1'}}]

更新行

Lance 支持使用 SQL 表达式更新行 lance.LanceDataset.update 方法。例如，如果我们注意到数据集中 Bob 的名字有时被写成 Blob ，我们可以这样修复：



In [17]:
import lance

dataset = lance.dataset("./alice_and_bob.lance")
dataset.update({"name": "'Bob'"}, where="name = 'Blob'")

{'num_rows_updated': 0}

In [22]:
dataset.to_table().to_pandas()

,name,age
0,Alice,20


更新值是 SQL 表达式，因此 'Bob' 用单引号括起来。这意味着我们可以根据需求使用引用现有列的复杂表达式。例如，如果两年过去了，我们想在同一个示例中更新 Alice 和 Bob 的年龄，可以这样写：

In [23]:
import lance

dataset = lance.dataset("./alice_and_bob.lance")
# age统一增加2岁
dataset.update({"age": "age + 2"})
dataset.to_table().to_pandas()

,name,age
0,Alice,22


如果您尝试使用新值更新一组单独的行，那么使用下面描述的合并插入操作通常更有效。



In [36]:
import lance
# shutil.rmtree("./alice_and_bob.lance", ignore_errors=True)
# Change the ages of both Alice and Bob
new_table = pa.Table.from_pylist([{"name": "Alice", "age": 30},
                                  {"name": "Bob", "age": 20}])

# This works, but is inefficient, see below for a better approach
dataset = lance.dataset("./alice_and_bob.lance")
for idx in range(new_table.num_rows):
  name = new_table[0][idx].as_py()
  new_age = new_table[1][idx].as_py()
  print(type(name), type(new_age))
  dataset.update({"age": new_age}, where=f"name='{name}'")

<class 'str'> <class 'int'>


TypeError: 'int' object cannot be converted to 'PyString'

Lance 支持合并插入操作。该操作可用于批量添加新数据，同时（可能）与现有数据进行匹配。此操作可用于多种不同的用例。

lance.LanceDataset.update 方法对于根据过滤器更新行非常有用。但是，如果我们想用新行替换现有行，则需要使用 lance.LanceDataset.merge_insert 操作会更有效率：


In [37]:
import lance

dataset = lance.dataset("./alice_and_bob.lance")
print(dataset.to_table().to_pandas())
#     name  age
# 0  Alice   20
# 1    Bob   30

# Change the ages of both Alice and Bob
new_table = pa.Table.from_pylist([{"name": "Alice", "age": 2},
                                  {"name": "Bob", "age": 3}])
# This will use `name` as the key for matching rows.  Merge insert
# uses a JOIN internally and so you typically want this column to
# be a unique key or id of some kind.
rst = dataset.merge_insert("name") \
       .when_matched_update_all() \
       .execute(new_table)
print(dataset.to_table().to_pandas())
#     name  age
# 0  Alice    2
# 1    Bob    3

    name  age
0  Alice   20
1    Bob   30
    name  age
0  Alice    2
1    Bob    3


有时我们只想插入之前未插入过的数据。例如，当我们有一批数据，但不知道之前添加过哪些行，并且不想创建重复的行时，就会发生这种情况。我们可以使用合并插入操作来实现这一点：

In [38]:
# Bob is already in the table, but Carla is new
new_table = pa.Table.from_pylist([{"name": "Bob", "age": 30},
                                  {"name": "Carla", "age": 37}])

dataset = lance.dataset("./alice_and_bob.lance")

# This will insert Carla but leave Bob unchanged
_ = dataset.merge_insert("name") \
       .when_not_matched_insert_all() \
       .execute(new_table)
# Verify that Carla was added but Bob remains unchanged
print(dataset.to_table().to_pandas())
#     name  age
# 0  Alice   20
# 1    Bob   30
# 2  Carla   37

    name  age
0  Alice    2
1    Bob    3
2  Carla   37


有时我们希望将上述两种行为结合起来。如果某行已存在，我们想更新它。如果该行不存在，我们想添加它。此操作有时称为“upsert”。我们也可以使用合并插入操作来实现这一点：

In [39]:
import lance
import pyarrow as pa

# Change Carla's age and insert David
new_table = pa.Table.from_pylist([{"name": "Carla", "age": 27},
                                  {"name": "David", "age": 42}])

dataset = lance.dataset("./alice_and_bob.lance")

# This will update Carla and insert David
_ = dataset.merge_insert("name") \
       .when_matched_update_all() \
       .when_not_matched_insert_all() \
       .execute(new_table)
# Verify the results
print(dataset.to_table().to_pandas())
#     name  age
# 0  Alice   20
# 1    Bob   30
# 2  Carla   27
# 3  David   42

    name  age
0  Alice    2
1    Bob    3
2  Carla   27
3  David   42


一个不太常见但仍然有用的行为是将现有行（由过滤器定义）的某些区域替换为新数据。这类似于在单个事务中同时执行删除和插入操作。例如：

In [40]:
import lance
import pyarrow as pa

new_table = pa.Table.from_pylist([{"name": "Edgar", "age": 46},
                                  {"name": "Francene", "age": 44}])

dataset = lance.dataset("./alice_and_bob.lance")
print(dataset.to_table().to_pandas())
#       name  age
# 0    Alice   20
# 1      Bob   30
# 2  Charlie   45
# 3    Donna   50

# This will remove anyone above 40 and insert our new data
_ = dataset.merge_insert("name") \
       .when_not_matched_insert_all() \
       .when_not_matched_by_source_delete("age >= 40") \
       .execute(new_table)
# Verify the results - people over 40 replaced with new data
print(dataset.to_table().to_pandas())
#        name  age
# 0     Alice   20
# 1       Bob   30
# 2     Edgar   46
# 3  Francene   44

    name  age
0  Alice    2
1    Bob    3
2  Carla   27
3  David   42
       name  age
0     Alice    2
1       Bob    3
2     Carla   27
3     Edgar   46
4  Francene   44


读取lance数据集

In [ ]:
import lance
# 从对象存储读取一个lance数据集
ds = lance.dataset("s3://bucket/path/imagenet.lance")
# Or local path
# 从本地读取一个lance数据集
ds = lance.dataset("./imagenet.lance")

In [60]:
#  读取 Lance 数据集最直接的方法是利用 lance.LanceDataset.to_table 方法将整个数据集加载到内存中。
ds = lance.dataset("./alice_and_bob.lance")
ds.versions()

[{'version': 1,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 24, 58, 757445),
  'metadata': {'total_data_file_rows': '2',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '593',
   'total_fragments': '1',
   'total_rows': '2'}},
 {'version': 2,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 26, 16, 478634),
  'metadata': {'total_data_file_rows': '2',
   'total_data_files': '1',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '593',
   'total_fragments': '1',
   'total_rows': '2'}},
 {'version': 3,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 27, 22, 546701),
  'metadata': {'total_data_file_rows': '3',
   'total_data_files': '2',
   'total_deletion_file_rows': '0',
   'total_deletion_files': '0',
   'total_files_size': '1186',
   'total_fragments': '2',
   'total_rows': '3'}},
 {'version': 4,
  'timestamp': datetime.datetime(2025, 10, 22, 14, 27, 47, 500157),


由于 Lance 是一种高性能的列式格式，它能够利用以下方式高效读取数据集的子集： 列（投影） 下推和过滤器（谓词） 下推。


In [61]:
ds.to_table().to_pandas()

,name,age
0,Alice,2
1,Bob,3
2,Carla,27
3,Edgar,46
4,Francene,44


In [59]:
table = ds.to_table(
    columns=["name", "age"],
    filter="age > 40 AND name in ('Edgar')",
    limit=1000,
    offset=0)
table.to_pandas()

ValueError: LanceError(Schema): Schema error: No field named age. Did you mean 'image'?., /home/runner/work/lance/lance/rust/lance/src/dataset/scanner.rs:259:37

如果数据集太大，内存无法容纳，可以使用 lance.LanceDataset.to_batches 方法批量读取：


不出所料， lance.LanceDataset.to_batches 采用与 lance.LanceDataset.to_table 函数相同的参数。



In [51]:
for batch in ds.to_batches(columns=["name","age"], filter="age > 10"):
    # do something with batch
    print(batch)

pyarrow.RecordBatch
name: string
age: int64
----
name: ["Carla","Edgar","Francene"]
age: [27,46,44]


In [57]:
import lance
import pyarrow as pa
import numpy as np

# ------------------------------
# 1. 准备示例数据
# ------------------------------
data = [
    {
        "image": np.random.randint(0, 256, size=(32, 32, 3), dtype=np.uint8).tolist(),
        "label": np.random.randint(0, 21)
    }
    for _ in range(100)
]

schema = pa.schema([
    pa.field("image", pa.list_(pa.list_(pa.list_(pa.uint8())))),
    pa.field("label", pa.int32())
])

table = pa.Table.from_pylist(data, schema=schema)

# ✅ 写入 Lance 数据集
lance.write_dataset(table, "./image_dataset.lance", mode="overwrite")

# ------------------------------
# 2. 批次读取并处理
# ------------------------------
ds = lance.dataset("./image_dataset.lance")

def compute_on_batch(batch: pa.RecordBatch):
    # 将 image 列转为 numpy 数组
    images = [np.array(img, dtype=np.uint8) for img in batch["image"].to_pylist()]
    images = np.stack(images)  # (batch_size, 32, 32, 3)

    gray_images = np.mean(images, axis=-1)
    avg_brightness = np.mean(gray_images, axis=(1, 2))

    print(f"批次大小: {len(images)}")
    print(f"平均亮度: {avg_brightness.round(2)}")
    print("------------------------")

# ✅ 使用 where 而非 filter
for batch in ds.to_batches(columns=["image", "label"], where="label = 10", batch_size=10):
    compute_on_batch(batch)


批次大小: 10
平均亮度: [126.74 129.15 126.13 127.15 127.2  127.78 127.87 127.09 127.9  125.83]
------------------------
批次大小: 10
平均亮度: [127.79 127.43 127.85 126.11 124.02 125.19 126.63 127.53 125.99 129.66]
------------------------
批次大小: 10
平均亮度: [126.94 128.8  128.04 128.5  128.85 127.17 125.4  127.43 125.64 126.46]
------------------------
批次大小: 10
平均亮度: [128.61 125.03 126.   127.62 128.66 128.15 126.83 126.18 127.87 128.59]
------------------------
批次大小: 10
平均亮度: [129.25 129.76 128.89 126.32 128.97 129.5  127.13 125.37 127.05 127.67]
------------------------
批次大小: 10
平均亮度: [128.13 128.41 127.03 128.91 127.75 127.55 125.49 128.93 130.19 128.75]
------------------------
批次大小: 10
平均亮度: [128.77 128.3  127.14 127.98 126.91 129.53 126.15 125.02 128.08 126.6 ]
------------------------
批次大小: 10
平均亮度: [129.09 124.26 129.51 129.94 128.29 129.18 127.32 128.04 124.89 125.46]
------------------------
批次大小: 10
平均亮度: [126.41 127.32 126.99 129.83 125.16 127.96 126.66 129.11 126.5  126.88]
-----------------

Lance 鼓励使用标准 SQL 表达式作为数据集过滤的谓词。通过将 SQL 谓词直接下推到存储系统，扫描期间的整体 I/O 负载显著降低。

目前，Lance 支持的表达方式越来越多。

```
>, >=, <, <=, =
AND, OR, NOT
IS NULL, IS NOT NULL
IS TRUE, IS NOT TRUE, IS FALSE, IS NOT FALSE
LIKE, NOT LIKE
regexp_match(column, pattern)
CAST
```
((label IN [10, 20]) AND (note['email'] IS NOT NULL)) OR NOT note['created']



嵌套字段可以通过下标访问。结构体字段可以使用字段名作为下标，列表字段可以使用索引作为下标。

如果列名包含特殊字符或 SQL 关键字 ，则可以使用反引号 ( ` ) 进行转义。对于嵌套字段，路径的每个部分都必须用反引号括起来。



```sql
`CUBE` = 10 AND `column name with space` IS NOT NULL AND `nested with space`.`inner with space` < 2
```
Field names containing periods (.) are not supported.
不支持包含句点 ( . ) 的字段名称。

日期、时间戳和小数的字面量可以通过在类型名称后写入字符串值来表示。例如



```sql
date_col = date '2021-01-01'
and timestamp_col = timestamp '2021-01-01 00:00:00'
and decimal_col = decimal(8,3) '1.000'
```




In [64]:
# 随机读取
#  读取 Lance 数据集最直接的方法是利用 lance.LanceDataset.to_table 方法将整个数据集加载到内存中。
ds = lance.dataset("./alice_and_bob.lance")
ds.versions()
table = ds.take([1,2,3], columns=['age'])
table.to_pandas()

,age
0,3
1,27
2,46


from matplotlib import pyplot as plt
_df_10['age'].plot(kind='hist', bins=20, title='age')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['age']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'age'}, axis=1)
              .sort_values('age', ascending=True))
  xs = counted['age']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_11.sort_values('age', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('age')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_12['age'].plot(kind='line', figsize=(8, 4), title='age')
plt.gca().spines[['top', 'right']].set_visible(False)

随着时间的推移，某些操作会导致 Lance 数据集布局不佳。例如，许多小的附加操作会导致大量小碎片。或者，删除大量行会导致查询速度变慢，因为需要过滤掉已删除的行。

为了解决这个问题，Lance 提供了优化数据集布局的方法。


可以重写数据文件，因此文件更少。当传递 target_rows_per_fragment 到 lance.dataset.DatasetOptimizer.compact_files ，Lance 将跳过任何已经超过该行数的片段，并重写其他片段。片段将根据其片段 ID 进行合并，因此将保留数据的固有顺序。（压缩会创建表的新版本。它不会删除旧版本的表及其引用的文件。）

In [71]:
import lance

dataset = lance.dataset("./alice_and_bob.lance")
dataset.optimize.compact_files(target_rows_per_fragment=1024 * 1024)

CompactionMetrics(fragments_removed=3, fragments_added=1, files_removed=4, files_added=1)

在压缩过程中，Lance 还可以删除已删除的行。重写的片段将没有删除文件。这可以提高扫描性能，因为在扫描期间不必跳过软删除的行。

重写文件时，原始行地址将失效。这意味着受影响的文件不再是任何 ANN 索引的一部分（如果它们以前是）。因此，建议在重新构建索引之前重写文件。
